# Web Scraping — Privacy Risk Scorer

This notebook collects, per URL:
- Tracker data (third-party domains, ad networks) — using Selenium network logs + Disconnect.me's verified tracker list
- Cookie consent signals (banner, reject button, CMP name) — using BeautifulSoup HTML parsing

Cleaned version: every function is now defined ONCE, in one place, to avoid
confusion from earlier exploration where things got redefined multiple times.

## Step 0 — All imports, in one place

In [1]:
import json
import time
import re
from urllib.parse import urlparse, urljoin

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup

import pandas as pd
import requests


## Step 1 — Driver creation
ONE version only, with logging enabled (needed for both trackers AND consent —
no need for two separate driver functions anymore).

In [2]:
def create_driver():
    """Creates a headless Chrome browser instance with network logging enabled."""
    options = Options()
    options.add_argument('--headless=new')
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument('--disable-gpu')
    options.add_argument('--window-size=1920,1080')
    options.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36')
    options.set_capability('goog:loggingPrefs', {'performance': 'ALL'})
    
    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=options)
    driver.set_page_load_timeout(15)
    return driver


def is_driver_alive(driver):
    """Checks if the driver session is still responsive."""
    try:
        driver.title
        return True
    except Exception:
        return False


def ensure_driver(driver):
    """Returns a working driver — recreates it if the current one has crashed."""
    if driver is None or not is_driver_alive(driver):
        try:
            driver.quit()
        except Exception:
            pass
        return create_driver()
    return driver

## Step 2 — Tracker detection setup
Load the Disconnect.me verified tracker list (replaces any hand-typed list).

In [3]:
with open('../raw_data/disconnect_services.json', 'r') as f:
    disconnect_data = json.load(f)


def build_domain_lookup(disconnect_data):
    """Flattens the nested Disconnect JSON into {domain: category}."""
    domain_to_category = {}
    for category, services in disconnect_data['categories'].items():
        for service in services:
            for company_name, urls_dict in service.items():
                for homepage, domains in urls_dict.items():
                    for domain in domains:
                        domain_to_category[domain] = category
    return domain_to_category


domain_to_category = build_domain_lookup(disconnect_data)
print(f'Loaded {len(domain_to_category)} known tracker domains')


def is_known_ad_network(domain):
    """Returns True if domain matches a known Advertising or Analytics tracker."""
    for known_domain, category in domain_to_category.items():
        if known_domain in domain and category in ('Advertising', 'Analytics'):
            return True
    return False

Loaded 4443 known tracker domains


## Step 3 — Tracker collection function

In [36]:
def extract_domains_from_logs(driver):
    """Reads Chrome's performance logs and extracts all unique domains contacted."""
    try:
        logs = driver.get_log('performance')
    except Exception:
        return set()
    
    domains = set()
    
    for entry in logs:
        try:
            message = json.loads(entry['message'])['message']
            if message.get('method') == 'Network.requestWillBeSent':
                url = message['params']['request']['url']
                domain = urlparse(url).netloc
                if domain:
                    domains.add(domain)
        except (KeyError, json.JSONDecodeError):
            continue
    
    return domains

def collect_trackers(driver, url):
    """
    Loads a URL and returns tracker-related features as a dictionary.
    """
    main_domain = urlparse(url).netloc.replace('www.', '')
    
    try:
        driver.get(url)
        time.sleep(5)  # let JavaScript fire trackers (bumped from 3s -> 5s, see validation notes)
    except Exception as e:
        return {
            'tracker_count': None,
            'ad_network_count': None,
            'tracker_error': str(e)
        }
    
    all_domains = extract_domains_from_logs(driver)
    third_party = [d for d in all_domains if main_domain not in d]
    ad_network_matches = [d for d in third_party if is_known_ad_network(d)]
    
    return {
        'tracker_count': len(third_party),
        'ad_network_count': len(ad_network_matches),
        'tracker_error': None
    }

## Step 4 — Cookie consent detection setup

In [5]:
COOKIE_BANNER_KEYWORDS = [
    'cookie', 'cookies', 'consent', 'gdpr', 'rgpd',
    'confidentialite', 'datenschutz', 'privacy'
]

REJECT_BUTTON_KEYWORDS = [
    'reject all', 'refuser', 'refuse all', 'decline',
    'tout refuser', 'alle ablehnen', 'ablehnen','continuer sans accepter'
]

ACCEPT_BUTTON_KEYWORDS = [
    'accept all', 'accepter', 'accept cookies',
    'tout accepter', 'alle akzeptieren'
]

# Known CMP vendor names — confirmed working via manual validation against
# BigQuery ground truth (Didomi confirmed on abeille-assurances.fr,
# Funding Choices confirmed on m.gamer.com.tw via fundingchoicesmessages.google.com)
KNOWN_CMP_NAMES = [
    'didomi', 'onetrust', 'cookiebot', 'usercentrics',
    'trustarc', 'klaro', 'osano', 'cookieyes',
    'complianz', 'iubenda', 'quantcast',
    'funding choices', 'fundingchoices',
    'setupad',  # found via manual check — not in original list
    'conversant',  # found in BigQuery validation (Conversant Consent Tool)
    'cookie notice',
]

## Step 5 — Cookie consent collection function

In [6]:
def collect_consent(driver, url):
    """
    Loads a URL and returns cookie consent features as a dictionary.
    """
    try:
        driver.get(url)
        time.sleep(5)  # let the cookie banner render (bumped from 3s -> 5s)
    except Exception as e:
        return {
            'has_cookie_banner': None,
            'has_reject_button': None,
            'has_accept_button': None,
            'cmp_detected': None,
            'consent_error': str(e)
        }
    
    html = driver.page_source
    soup = BeautifulSoup(html, 'html.parser')
    page_text = soup.get_text().lower()
    html_lower = html.lower()
    
    has_banner = any(kw in page_text for kw in COOKIE_BANNER_KEYWORDS)
    has_reject = any(kw in page_text for kw in REJECT_BUTTON_KEYWORDS)
    has_accept = any(kw in page_text for kw in ACCEPT_BUTTON_KEYWORDS)
    
    detected_cmp = None
    for cmp_name in KNOWN_CMP_NAMES:
        if cmp_name in html_lower:
            detected_cmp = cmp_name
            break
    
    return {
        'has_cookie_banner': has_banner,
        'has_reject_button': has_reject,
        'has_accept_button': has_accept,
        'cmp_detected': detected_cmp,
        'consent_error': None
    }

## Step 6 — Quick test on known sites before the full run
One driver, used for both functions — no need to create/quit repeatedly.

In [7]:
driver = create_driver()

test_sites = ['https://www.lemonde.fr/', 'https://www.abeille-assurances.fr/']

for site in test_sites:
    driver = ensure_driver(driver)  # safety check before each site
    
    tracker_result = collect_trackers(driver, site)
    consent_result = collect_consent(driver, site)
    
    print(site)
    print(' trackers:', tracker_result)
    print(' consent: ', consent_result)
    print()

driver.quit()

https://www.lemonde.fr/
 trackers: {'tracker_count': 4, 'ad_network_count': 4, 'tracker_error': None}
 consent:  {'has_cookie_banner': True, 'has_reject_button': True, 'has_accept_button': True, 'cmp_detected': None, 'consent_error': None}

https://www.abeille-assurances.fr/
 trackers: {'tracker_count': 12, 'ad_network_count': 7, 'tracker_error': None}
 consent:  {'has_cookie_banner': True, 'has_reject_button': True, 'has_accept_button': True, 'cmp_detected': 'didomi', 'consent_error': None}



## Step 7 — Validation against BigQuery ground truth
(Kept from your earlier work — this is the validation that found the
Funding Choices timing issue and the Setupad/custom-banner cases.)

In [8]:
df_cmp = pd.read_csv('../raw_data/bigquery_cmp_names.csv')

validation_sample = df_cmp.dropna(subset=['cmp_name']).sample(10, random_state=42)
print(validation_sample[['page', 'cmp_name']])

                              page                 cmp_name
1033     https://www.haberler.com/          Funding Choices
309       https://www.dochord.com/          Funding Choices
1343     https://it.investing.com/  Conversant Consent Tool
1407            https://ezgif.com/  Conversant Consent Tool
680       https://modilimitado.io/            Cookie Notice
1531  https://www.abercrombie.com/                 OneTrust
670        https://m.gamer.com.tw/          Funding Choices
275     https://www.glassdoor.com/                 OneTrust
497         https://sportsport.ba/          Funding Choices
65         https://www.ntv.com.tr/                 OneTrust


In [9]:
driver = create_driver()

validation_results = []

for _, row in validation_sample.iterrows():
    driver = ensure_driver(driver)  # recreate if crashed on previous URL
    
    url = row['page']
    expected_cmp = row['cmp_name']
    
    scraper_result = collect_consent(driver, url)
    found_cmp = scraper_result['cmp_detected']
    
    # A match counts if EITHER tool's name appears related to the other
    # (loosened from exact match, since sites can run multiple CMPs at once)
    match = False
    if found_cmp:
        match = (found_cmp.lower() in str(expected_cmp).lower()) or (str(expected_cmp).lower() in found_cmp.lower())
    
    validation_results.append({
        'url': url,
        'bigquery_cmp': expected_cmp,
        'scraper_cmp': found_cmp,
        'match': match
    })

driver.quit()

df_validation = pd.DataFrame(validation_results)
print(df_validation)
print()
print(f"Agreement rate: {df_validation['match'].sum()} / {len(df_validation)}")

                            url             bigquery_cmp     scraper_cmp  \
0     https://www.haberler.com/          Funding Choices        trustarc   
1      https://www.dochord.com/          Funding Choices  fundingchoices   
2     https://it.investing.com/  Conversant Consent Tool        onetrust   
3            https://ezgif.com/  Conversant Consent Tool         setupad   
4      https://modilimitado.io/            Cookie Notice   cookie notice   
5  https://www.abercrombie.com/                 OneTrust        onetrust   
6       https://m.gamer.com.tw/          Funding Choices        trustarc   
7    https://www.glassdoor.com/                 OneTrust        onetrust   
8        https://sportsport.ba/          Funding Choices          didomi   
9       https://www.ntv.com.tr/                 OneTrust        onetrust   

   match  
0  False  
1  False  
2  False  
3  False  
4   True  
5   True  
6  False  
7   True  
8  False  
9   True  

Agreement rate: 4 / 10


##### Debugging
there is one website that doesn't have a sms detected by the scrpper. However when i open the website it has a consent notice by Didomi , so let's debug this 

In [10]:
driver = create_driver()


In [11]:
driver.quit()

In [12]:
driver = ensure_driver(driver)
driver.get('https://sportsport.ba/')
time.sleep(5)
html = driver.page_source.lower()

print('didomi' in html)
print(len(html))  # sanity check — did the page actually load fully?

True
353742


In [13]:
driver.get('https://sportsport.ba/')
time.sleep(5)

html = driver.page_source
html_lower = html.lower()

# Manually repeat exactly what collect_consent() does internally
detected_cmp = None
for cmp_name in KNOWN_CMP_NAMES:
    if cmp_name in html_lower:
        detected_cmp = cmp_name
        print(f'Found: {cmp_name}')
        break

print('Final detected_cmp:', detected_cmp)
print()
print('Is didomi literally in KNOWN_CMP_NAMES?', 'didomi' in KNOWN_CMP_NAMES)
print('Full list:', KNOWN_CMP_NAMES)

Found: didomi
Final detected_cmp: didomi

Is didomi literally in KNOWN_CMP_NAMES? True
Full list: ['didomi', 'onetrust', 'cookiebot', 'usercentrics', 'trustarc', 'klaro', 'osano', 'cookieyes', 'complianz', 'iubenda', 'quantcast', 'funding choices', 'fundingchoices', 'setupad', 'conversant', 'cookie notice']


In [14]:
driver = ensure_driver(driver)
result = collect_consent(driver, 'https://sportsport.ba/')
print(result)

{'has_cookie_banner': True, 'has_reject_button': True, 'has_accept_button': True, 'cmp_detected': 'didomi', 'consent_error': None}


In [15]:
driver.quit()

so it seems the issue is the time out the page takes too long to load. so for now we accept if a small percentage of out data is not matched

## Step 8 - Privacy policy detection and scoring

What this does, in 3 steps:
1. Find the actual privacy policy LINK on the page (not just check if the word exists)
2. Visit that link and get the real policy text
3. Count how many "risky" phrases appear (data selling, sharing with third parties)

- Keywords we search for

In [16]:
# Words that suggest a link IS a privacy policy link
POLICY_LINK_WORDS = [
    # English
    'privacy policy', 'privacy notice',
    # French
    'politique de confidentialite', 'politique de confidentialité', 'mentions legales', 'mentions légales',
    # German
    'datenschutz', 'datenschutzerklarung', 'datenschutzerklärung',
    # Spanish
    'politica de privacidad', 'política de privacidad', 'aviso de privacidad',
    # Italian
    'informativa sulla privacy', 'politica sulla privacy',
    # Dutch
    'privacybeleid', 'privacyverklaring',
    # Generic fallback
    'privacy', 'cookies'
]

# Risky phrases to count once we're ON the policy page
RISKY_PHRASES = [
    'sell your data', 'sell your information',
    'third party', 'third-party',
    'share your information', 'partager', 'partenaires',
    'indefinitely', 'durée indéterminée'
]

### Function 1 - find the privacy policy link on the page
Looks through every link (`<a>` tag) on the page and checks if its
text matches one of our keywords.

In [27]:
def find_policy_link(driver, page_url):
    """Looks for a privacy policy link on the current page. Returns the URL or None."""
    try:
        html = driver.page_source
    except Exception:
        return None
    
    soup = BeautifulSoup(html, 'html.parser')
    all_links = soup.find_all('a', href=True)
    
    matches = {}
    for link in all_links:
        link_text = link.get_text().lower()
        for word in POLICY_LINK_WORDS:
            if word in link_text:
                relative_url = link['href']
                full_url = urljoin(page_url, relative_url)
                matches[word] = full_url
    
    for word in POLICY_LINK_WORDS:
        if word in matches:
            return matches[word]
    
    return None


### Function 2 - visit the policy page and count risky phrases
Uses requests instead of Selenium since policy pages are simple text pages.

In [28]:
def score_policy_page(driver, policy_url):
    """Visits the policy URL using Selenium and counts risky phrases."""
    
    if policy_url is None:
        return {
            'has_privacy_policy': False,
            'policy_word_count': 0,
            'risky_phrase_count': 0
        }
    
    # Skip anything that isn't a real http/https link
    if not policy_url.startswith('http'):
        return {
            'has_privacy_policy': False,
            'policy_word_count': 0,
            'risky_phrase_count': 0
        }
    
    driver.get(policy_url)
    time.sleep(3)
    
    soup = BeautifulSoup(driver.page_source, 'html.parser')
    text = soup.get_text().lower()
    
    word_count = len(text.split())
    
    risky_count = 0
    for phrase in RISKY_PHRASES:
        risky_count += text.count(phrase)
    
    return {
        'has_privacy_policy': True,
        'policy_word_count': word_count,
        'risky_phrase_count': risky_count
    }

### Function 3 - put it together
This is the one you call from your main pipeline.

In [29]:
def collect_policy(driver, page_url):
    """Just checks if a privacy policy link exists — doesn't visit the page."""
    link = find_policy_link(driver, page_url)
    return {'has_privacy_policy': link is not None}

In [30]:
driver = create_driver()

driver.get('https://matomo.org/')
time.sleep(3)

result = collect_policy(driver, 'https://matomo.org/')
print(result)

driver.quit()

{'has_privacy_policy': True}


## Step 9 - Combine everything into one function

Calls collect_trackers, collect_consent, and collect_policy for one URL,
and merges all three results into a single dictionary (one row of data).

In [37]:
def collect_all(driver, url):
    try:
        trackers = collect_trackers(driver, url)
        consent = collect_consent(driver, url)
        policy = collect_policy(driver, url)
        
        row = {'url': url}
        row.update(trackers)
        row.update(consent)
        row.update(policy)
        return row
    except Exception as e:
        return {'url': url, 'error': str(e)}

In [32]:
driver = create_driver()

result = collect_all(driver, 'https://matomo.org/')
print(result)

driver.quit()

{'url': 'https://matomo.org/', 'tracker_count': 8, 'ad_network_count': 1, 'tracker_error': None, 'has_cookie_banner': True, 'has_reject_button': False, 'has_accept_button': False, 'cmp_detected': 'usercentrics', 'consent_error': None, 'has_privacy_policy': True}


## Step 10 - Run on all URLs and save results

This loops through your full urls.csv, runs collect_all on each one,
and saves progress every 50 sites (so you don't lose everything if
something crashes partway through a long run).

In [23]:
df_urls = pd.read_csv('../clean_data/urls.csv')
url_list = df_urls['url'].tolist()

print(f'Total URLs to scrape: {len(url_list)}')

Total URLs to scrape: 1500


### Test run on just 5 URLs first
Always test small before committing to the full run.

In [24]:
driver = create_driver()

test_results = []
for url in url_list[:5]:
    print('Scraping:', url)
    row = collect_all(driver, url)
    test_results.append(row)

driver.quit()

df_test = pd.DataFrame(test_results)
print(df_test)

Scraping: https://youtu.be
Scraping: https://europa.eu
Scraping: https://telekom.de
Scraping: https://iiko.it
Scraping: https://nominetdns.uk
                     url  tracker_count  ad_network_count  \
0       https://youtu.be            8.0               2.0   
1      https://europa.eu            9.0               2.0   
2     https://telekom.de            9.0               4.0   
3        https://iiko.it            NaN               NaN   
4  https://nominetdns.uk            NaN               NaN   

                                       tracker_error has_cookie_banner  \
0                                               None              True   
1                                               None              True   
2                                               None              True   
3  Message: unknown error: net::ERR_NAME_NOT_RESO...              None   
4  Message: unknown error: net::ERR_NAME_NOT_RESO...              None   

  has_reject_button has_accept_button cmp_dete

### The full run
This will take a while (roughly 1-2 minutes per site x 1500 sites).
Run this overnight or while doing something else.

In [38]:
driver = create_driver()

all_results = []

for i, url in enumerate(url_list[561:]):
    
    driver = ensure_driver(driver)  # recreate if it crashed on a previous site
    
    row = collect_all(driver, url)
    all_results.append(row)
    
    if (i + 1) % 100 == 0:
        print(f'Done {i+1} / {len(url_list)}')
    
    # Save progress every 50 sites, just in case
    if (i + 1) % 100 == 0:
        pd.DataFrame(all_results).to_csv('../raw_data/scraped_results_progress2.csv', index=False)

driver.quit()

print('Scraping finished')

Done 100 / 1500
Done 200 / 1500
Done 300 / 1500
Done 400 / 1500
Done 500 / 1500
Done 600 / 1500
Done 700 / 1500
Done 800 / 1500
Done 900 / 1500
Scraping finished


In [39]:
df1 = pd.read_csv('../raw_data/scraped_results_progress.csv')
df2 = pd.read_csv('../raw_data/scraped_results_progress2.csv')
df_final = pd.concat([df1, df2], ignore_index=True)
df_final.to_csv('../raw_data/scraped_results.csv', index=False)
print(df_final.shape)

(1400, 10)


# Cleaning scraped_results.csv

In [ ]:
import pandas as pd

df = pd.read_csv('../raw_data/scraped_results.csv')
print('Starting shape:', df.shape)
df.head()

Starting shape: (1400, 10)


,url,tracker_count,ad_network_count,tracker_error,has_cookie_banner,has_reject_button,has_accept_button,cmp_detected,consent_error,has_privacy_policy
0,https://youtu.be,9.0,3.0,NaN,True,True,True,NaN,NaN,True
1,https://europa.eu,10.0,2.0,NaN,True,False,True,NaN,NaN,True
2,https://telekom.de,9.0,4.0,NaN,True,False,True,NaN,NaN,True
3,https://iiko.it,NaN,NaN,Message: unknown error: net::ERR_NAME_NOT_RESO...,NaN,NaN,NaN,NaN,Message: unknown error: net::ERR_NAME_NOT_RESO...,False
4,https://nominetdns.uk,NaN,NaN,Message: unknown error: net::ERR_NAME_NOT_RESO...,NaN,NaN,NaN,NaN,Message: unknown error: net::ERR_NAME_NOT_RESO...,False


## Step 1 - Drop dead sites (failed to load entirely)

In [ ]:
before = len(df)

df = df[df['tracker_count'].notna()].copy()

print(f'Dropped {before - len(df)} dead sites')
print(f'Remaining: {len(df)} rows')

Dropped 248 dead sites
Remaining: 1152 rows


## Step 2 - Drop the error columns
They only existed to help debug during scraping.

In [ ]:
df = df.drop(columns=['tracker_error', 'consent_error'])
print(df.columns.tolist())

['url', 'tracker_count', 'ad_network_count', 'has_cookie_banner', 'has_reject_button', 'has_accept_button', 'cmp_detected', 'has_privacy_policy']


## Step 3 - Fix number types

In [ ]:
df['tracker_count'] = df['tracker_count'].astype(int)
df['ad_network_count'] = df['ad_network_count'].astype(int)

print(df[['tracker_count', 'ad_network_count']].dtypes)

tracker_count       int64
ad_network_count    int64
dtype: object


## Step 4 - Fix boolean column types

In [ ]:
for col in ['has_cookie_banner', 'has_reject_button', 'has_accept_button']:
    df[col] = df[col].astype(bool)

print(df[['has_cookie_banner', 'has_reject_button', 'has_accept_button']].dtypes)

has_cookie_banner    bool
has_reject_button    bool
has_accept_button    bool
dtype: object


## Step 5 - Fill missing CMP names with 'none'
Missing here means no recognized vendor was found (often a custom banner).

In [ ]:
df['cmp_detected'] = df['cmp_detected'].fillna('none')

print(df['cmp_detected'].value_counts().head(10))

cmp_detected
none              747
onetrust          145
didomi             90
cookiebot          36
trustarc           30
usercentrics       24
iubenda            19
cookie notice      18
fundingchoices     15
klaro               7
Name: count, dtype: int64


## Step 6 - Check for duplicate URLs

In [ ]:
duplicate_count = df['url'].duplicated().sum()
print(f'Duplicate URLs found: {duplicate_count}')

df = df.drop_duplicates(subset='url')
print(f'Shape after dropping duplicates: {df.shape}')

Duplicate URLs found: 0
Shape after dropping duplicates: (1152, 8)


## Step 7 - Sanity check tracker_count distribution

In [ ]:
print(df['tracker_count'].describe())
print()
print('Top 5 highest tracker counts:')
print(df.nlargest(5, 'tracker_count')[['url', 'tracker_count']])

count    1152.000000
mean       21.208333
std        14.646782
min         1.000000
25%        11.000000
50%        18.000000
75%        28.000000
max       109.000000
Name: tracker_count, dtype: float64

Top 5 highest tracker counts:
                     url  tracker_count
42    https://elmundo.es            109
47      https://idnes.cz             93
609    https://record.pt             90
41       https://onet.pl             82
606  https://gazzetta.gr             81


## Step 8 - Final check and save

In [ ]:
print('=== Final summary ===')
print('Shape:', df.shape)
print()
print('Dtypes:')
print(df.dtypes)
print()
print('Nulls remaining:')
print(df.isnull().sum())
print()
df.head(10)

=== Final summary ===
Shape: (1152, 8)

Dtypes:
url                   object
tracker_count          int64
ad_network_count       int64
has_cookie_banner       bool
has_reject_button       bool
has_accept_button       bool
cmp_detected          object
has_privacy_policy      bool
dtype: object

Nulls remaining:
url                   0
tracker_count         0
ad_network_count      0
has_cookie_banner     0
has_reject_button     0
has_accept_button     0
cmp_detected          0
has_privacy_policy    0
dtype: int64



,url,tracker_count,ad_network_count,has_cookie_banner,has_reject_button,has_accept_button,cmp_detected,has_privacy_policy
0,https://youtu.be,9,3,True,True,True,none,True
1,https://europa.eu,10,2,True,False,True,none,True
2,https://telekom.de,9,4,True,False,True,none,True
5,https://t-online.de,14,4,True,False,False,none,True
6,https://bbc.co.uk,33,16,True,False,False,none,True
7,https://www.gov.uk,23,13,True,False,False,none,True
8,https://amazon.co.uk,5,2,True,True,False,cookie notice,True
10,https://dailymail.co.uk,35,19,True,False,False,trustarc,True
11,https://amazon.de,30,12,True,True,False,cookie notice,True
14,https://google.de,16,8,True,True,True,none,True


In [ ]:
df.to_csv('../clean_data/scraped_results_clean.csv', index=False)
print(f'Saved scraped_results_clean.csv — {len(df)} rows')

Saved scraped_results_clean.csv — 1152 rows


In [ ]:
df_none = df[df['cmp_detected'] == 'none']
print(df_none['url'].sample(5, random_state=1).tolist())

['https://ip-api.com', 'https://rae.es', 'https://alza.cz', 'https://tu-dresden.de', 'https://openai.com']
